# 01 — Frozen Split Protocol and Pre-Tournament Team-Season Snapshots

This notebook is the **scientific contract** for all later modeling. It sits between raw-data preparation and predictive feature engineering.

It does five things:

1. freezes the train/validation/benchmark design before model results can influence it;
2. builds deterministic Selection-Sunday snapshots from regular-season data only;
3. creates separate men's, women's, and pooled-comparison fold manifests;
4. carries detailed-data coverage and possession-quality warnings forward instead of silently deleting them;
5. runs leakage, coverage, routing, and reproducibility diagnostics that must pass before notebook `02` begins.

It deliberately does **not** fit Elo, opponent-adjusted ratings, XGBoost, LightGBM, calibrators, or ensembles. Those operations contain learned parameters and must be evaluated under the frozen folds created here.

> **Core rule:** every tournament game in season `Y` receives features frozen from season `Y` regular-season information available by DayNum 132. Earlier NCAA tournament games from season `Y` are never used to predict later NCAA tournament games in season `Y`.

## Why this design is stronger than simply copying a winning notebook

Publicly documented top solutions provide valuable ingredients, but no single public solution is a complete scientific template:

| Public result | Design worth retaining | What this project adds |
|---|---|---|
| 1st place | Separate men's/women's feature sets; seed prior; possession-based custom strength; shallow XGBoost; season-held-out predictions; isotonic calibration | Strict rolling-origin primary evaluation, nested selection, explicit quality propagation, and a locked recent benchmark |
| 2nd place | Rich efficiency/Elo/ranking differences and XGBoost + logistic-regression diversity | Matched separate-versus-pooled evaluation and fold-selected blend weights rather than an assumed fixed blend |
| 4th place | XGBoost + LightGBM, symmetric matchup augmentation, and season-grouped validation | Training seasons must precede the validation season in the primary protocol |
| 10th/19th places | Point-margin regression followed by probability calibration; multi-seed diversity; strong opponent-adjusted external metrics | Direct classification and margin regression will be compared under the same nested folds; external data must be timestamped and reproducible |
| 21st place | Walk-forward evaluation and gender-specific ensemble weights | Weight learning must occur within nested out-of-fold predictions and survive the locked benchmark |

This project therefore stands on the public work while adding:

- separate men's and women's primary pipelines;
- a mandatory pooled common-feature challenger;
- a partial-pooling ensemble candidate;
- expanding-window outer folds and nested expanding-window inner folds;
- a secondary LOSO benchmark only for comparability;
- prequential and static-block locked-benchmark protocols;
- explicit source-availability, coverage, and possession-quality diagnostics;
- machine-checkable leakage assertions and artifact fingerprints.

Public references retained for later comparison:

- [2026 first-place solution](https://www.kaggle.com/competitions/march-machine-learning-mania-2026/writeups/march-machine-learning-mania-2026-1st-place-solut)
- [2026 second-place solution](https://www.kaggle.com/competitions/march-machine-learning-mania-2026/writeups/2nd-place-solution-for-the-march-machine-learning)
- [2026 fourth-place solution](https://www.kaggle.com/competitions/march-machine-learning-mania-2026/writeups/4th-place-solution-for-the-march-machine-learning)
- [2026 tenth-place margin-regression solution](https://www.kaggle.com/competitions/march-machine-learning-mania-2026/writeups/march-mania-2026-10th-place-solution)
- [2026 nineteenth-place calibrated margin ensemble](https://www.kaggle.com/c/march-machine-learning-mania-2026/writeups/march-machine-learning-mania-2026-silver-medal-s)
- [2026 twenty-first-place gender-specific ensemble](https://www.kaggle.com/competitions/march-machine-learning-mania-2026/writeups/21st-place-solution-5-model-ensemble-with-optuna)

Only public writeups and public code are synthesized. Private notebooks, unpublished trials, and unavailable external datasets cannot be audited or claimed as reviewed.

The public writeups are evidence and inspiration—not permission to leak future seasons, tune on the benchmark, or assume that one competition's best feature set is universally optimal.


## Experimental architecture frozen here

The primary production architecture is:

```text
Men's snapshots   → men's feature store   → men's models   → men's calibrator/ensemble
Women's snapshots → women's feature store → women's models → women's calibrator/ensemble
```

The final submission will route rows by `Gender`. Two challengers are also preserved:

```text
Pooled common-feature model with Gender interactions
Partial-pooling blend of gender-specific and pooled predictions
```

The recent seasons 2022–2025 are labeled **locked benchmark seasons**. They are not a perfectly pristine scientific test because public competition results and writeups are now known, but they will be excluded from feature selection, tuning, calibration choice, and ensemble-weight optimization from this point forward. The primary development estimate comes from expanding-window outer folds ending in 2021.

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from march_mania.paths import get_project_paths

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)

PATHS = get_project_paths()
ROOT = PATHS.root
INTERIM = PATHS.interim
CONFIG_DIR = ROOT / "configs"
MODELING_REPORTS = ROOT / "reports" / "modeling"
FIGURE_DIR = ROOT / "reports" / "figures"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
MODELING_REPORTS.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Interim data:", INTERIM)
print("Python:", sys.executable)
print("Python version:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

assert "ml-modeling" in str(sys.executable).lower(), (
    "Select the Python (ml-modeling) kernel before continuing."
)

## 1. Load and validate notebook `00` outputs

The notebook refuses to continue if any required canonical table is missing. It reads only Parquet outputs created by notebook `00`; raw CSVs remain untouched.

In [ ]:
REQUIRED_TABLES = {
    "games_compact": "games_compact_canonical.parquet",
    "tournament_targets": "tournament_targets.parquet",
    "games_detailed": "games_detailed_team_long.parquet",
    "submission_matchups": "submission_matchups.parquet",
    "teams": "teams.parquet",
    "seeds": "seeds.parquet",
    "team_conferences": "team_conferences.parquet",
}

missing_files = [filename for filename in REQUIRED_TABLES.values() if not (INTERIM / filename).exists()]
assert not missing_files, (
    "Notebook 00 outputs are missing. Run notebooks/00_data_audit_and_preparation.ipynb first. "
    f"Missing: {missing_files}"
)

frames = {
    name: pd.read_parquet(INTERIM / filename)
    for name, filename in REQUIRED_TABLES.items()
}

games_compact = frames["games_compact"]
tournament_targets = frames["tournament_targets"]
games_detailed = frames["games_detailed"]
submission_matchups = frames["submission_matchups"]
teams = frames["teams"]
seeds = frames["seeds"]
team_conferences = frames["team_conferences"]

inventory = pd.DataFrame(
    [
        {
            "Table": name,
            "Rows": len(frame),
            "Columns": frame.shape[1],
            "MemoryMB": round(frame.memory_usage(index=True, deep=True).sum() / 1024**2, 3),
        }
        for name, frame in frames.items()
    ]
).sort_values("Table").reset_index(drop=True)

required_columns = {
    "games_compact": {
        "Gender", "GameType", "Season", "DayNum", "NumOT", "Team1ID", "Team2ID",
        "Team1Score", "Team2Score", "Team1Win", "Team1Margin", "Team1Loc", "GameKey",
    },
    "tournament_targets": {
        "Gender", "Season", "DayNum", "Team1ID", "Team2ID", "Team1Win", "Team1Margin", "GameKey",
    },
    "games_detailed": {
        "Gender", "GameType", "Season", "DayNum", "GameKey", "TeamID", "OppTeamID",
        "TeamScore", "OppScore", "Win", "TeamLoc", "Margin",
    },
    "submission_matchups": {
        "SubmissionFile", "ID", "Season", "Team1ID", "Team2ID", "Gender",
    },
}

for name, expected in required_columns.items():
    missing = expected.difference(frames[name].columns)
    assert not missing, f"{name} is missing required columns: {sorted(missing)}"

inventory

## 2. Freeze the split contract before modeling

Two feature universes are preserved:

- **Compact universe:** supports long-history seed, score, location, margin, and Elo baselines.
- **Rich universe:** begins when detailed box scores become available and supports possession/efficiency features.

The richer history is different by gender, so the fold definitions are also different. The pooled common-feature challenger uses only validation seasons available to both rich universes.

In [ ]:
SPLIT_CONTRACT: dict[str, Any] = {
    "contract_version": 1,
    "target_season": 2026,
    "feature_cutoff_day": 132,
    "locked_benchmark_seasons": [2022, 2023, 2024, 2025],
    "excluded_target_seasons": [2020],
    "development_last_season": 2021,
    "locked_benchmark_modes": {
        "primary": "prequential_refit_with_recipe_frozen_after_2021",
        "sensitivity": "static_target_model_fit_through_2021",
    },
    "primary_architecture": "separate_gender_models",
    "challenger_architectures": [
        "pooled_common_feature_model",
        "partial_pooling_blend",
    ],
    "universes": {
        "compact": {
            "M": {"first_season": 1985, "minimum_prior_training_seasons": 10},
            "W": {"first_season": 1998, "minimum_prior_training_seasons": 8},
        },
        "rich": {
            "M": {"first_season": 2003, "minimum_prior_training_seasons": 8},
            "W": {
                "first_season": 2010,
                "minimum_prior_training_seasons": 6,
                "complete_detailed_coverage_from": 2013,
            },
        },
    },
    "nested_tuning": {
        "maximum_inner_validation_seasons": 4,
        "minimum_inner_prior_seasons": {"M": 4, "W": 3, "Pooled": 3},
    },
    "evaluation": {
        "primary_selection_metric": "macro_mean_outer_season_brier",
        "secondary_selection_metric": "game_weighted_outer_brier",
        "mandatory_reports": [
            "brier_by_season",
            "brier_by_gender",
            "log_loss",
            "roc_auc",
            "calibration_intercept",
            "calibration_slope",
            "reliability_curve",
            "season_clustered_uncertainty",
        ],
    },
    "quality_thresholds": {
        "absolute_possession_gap_warning": 7.0,
        "minimum_detailed_coverage_for_rich_features": 0.90,
    },
    "rules": [
        "Tournament seasons are indivisible groups.",
        "Every training season must be earlier than its validation season.",
        "Locked benchmark seasons are excluded from all development folds.",
        "Same-season NCAA outcomes never enter pre-tournament features.",
        "Any learned preprocessing is fit inside the relevant training fold.",
        "Men and women have separate primary models and calibrators.",
        "Pooled models use only common features and are challengers, not assumed winners.",
    ],
}

contract_json = json.dumps(SPLIT_CONTRACT, sort_keys=True, separators=(",", ":"))
contract_hash = hashlib.sha256(contract_json.encode("utf-8")).hexdigest()
SPLIT_CONTRACT["contract_sha256"] = contract_hash

split_config_path = CONFIG_DIR / "splits.yaml"
if split_config_path.exists():
    existing = yaml.safe_load(split_config_path.read_text(encoding="utf-8"))
    existing_without_hash = dict(existing)
    existing_without_hash.pop("contract_sha256", None)
    proposed_without_hash = dict(SPLIT_CONTRACT)
    proposed_without_hash.pop("contract_sha256", None)
    assert existing_without_hash == proposed_without_hash, (
        "configs/splits.yaml already exists and differs from this notebook. "
        "Do not silently alter a frozen split contract; review and version the change explicitly."
    )
else:
    split_config_path.write_text(
        yaml.safe_dump(SPLIT_CONTRACT, sort_keys=False),
        encoding="utf-8",
    )

(MODELING_REPORTS / "split_contract.json").write_text(
    json.dumps(SPLIT_CONTRACT, indent=2),
    encoding="utf-8",
)

print("Split contract:", split_config_path)
print("Contract SHA-256:", contract_hash)
print(yaml.safe_dump(SPLIT_CONTRACT, sort_keys=False))

## 3. Hard temporal boundary and target audit

The feature cutoff is DayNum 132. The notebook removes nothing silently: it asserts that the modeling source contains only regular-season rows through that cutoff.

If a later Kaggle data refresh contains 2026 tournament outcomes, they are written to a quarantined local artifact and removed from all development tables. They must never enter feature construction, model selection, calibration, or blending for the 2026 replay.


In [ ]:
TARGET_SEASON = int(SPLIT_CONTRACT["target_season"])
CUTOFF_DAY = int(SPLIT_CONTRACT["feature_cutoff_day"])
LOCKED_BENCHMARK = set(SPLIT_CONTRACT["locked_benchmark_seasons"])
EXCLUDED_TARGET_SEASONS = set(SPLIT_CONTRACT["excluded_target_seasons"])
DEVELOPMENT_LAST_SEASON = int(SPLIT_CONTRACT["development_last_season"])

regular_compact = games_compact.loc[
    games_compact["GameType"].eq("Regular")
    & games_compact["DayNum"].le(CUTOFF_DAY)
].copy()

regular_detailed = games_detailed.loc[
    games_detailed["GameType"].eq("Regular")
    & games_detailed["DayNum"].le(CUTOFF_DAY)
].copy()

post_cutoff_regular_compact = games_compact.loc[
    games_compact["GameType"].eq("Regular")
    & games_compact["DayNum"].gt(CUTOFF_DAY)
].copy()

quarantined_target_outcomes = tournament_targets.loc[
    tournament_targets["Season"].ge(TARGET_SEASON)
].copy()
if not quarantined_target_outcomes.empty:
    quarantine_path = INTERIM / "quarantined_target_or_future_tournament_outcomes.parquet"
    quarantined_target_outcomes.to_parquet(
        quarantine_path,
        index=False,
        compression="zstd",
    )
    print(
        f"QUARANTINED {len(quarantined_target_outcomes):,} target/future tournament rows: "
        f"{quarantine_path}"
    )

tournament_targets = tournament_targets.loc[
    tournament_targets["Season"].lt(TARGET_SEASON)
].copy()

assert not regular_compact.empty
assert not regular_detailed.empty
assert regular_compact["DayNum"].max() <= CUTOFF_DAY
assert regular_detailed["DayNum"].max() <= CUTOFF_DAY
assert post_cutoff_regular_compact.empty
assert regular_compact["GameType"].eq("Regular").all()
assert regular_detailed["GameType"].eq("Regular").all()
assert tournament_targets["GameType"].eq("NCAA").all()
assert tournament_targets["Season"].lt(TARGET_SEASON).all()
assert tournament_targets["GameKey"].is_unique
assert not tournament_targets["Season"].isin(EXCLUDED_TARGET_SEASONS).any()
assert (tournament_targets["Team1ID"] < tournament_targets["Team2ID"]).all()

boundary_summary = pd.DataFrame(
    {
        "Dataset": [
            "regular_compact",
            "regular_detailed",
            "historical_tournament_targets",
            "quarantined_target_or_future_outcomes",
        ],
        "Rows": [
            len(regular_compact),
            len(regular_detailed),
            len(tournament_targets),
            len(quarantined_target_outcomes),
        ],
        "MinSeason": [
            regular_compact.Season.min(),
            regular_detailed.Season.min(),
            tournament_targets.Season.min(),
            quarantined_target_outcomes.Season.min() if not quarantined_target_outcomes.empty else np.nan,
        ],
        "MaxSeason": [
            regular_compact.Season.max(),
            regular_detailed.Season.max(),
            tournament_targets.Season.max(),
            quarantined_target_outcomes.Season.max() if not quarantined_target_outcomes.empty else np.nan,
        ],
        "MaxDayNum": [
            regular_compact.DayNum.max(),
            regular_detailed.DayNum.max(),
            tournament_targets.DayNum.max(),
            quarantined_target_outcomes.DayNum.max() if not quarantined_target_outcomes.empty else np.nan,
        ],
    }
)
boundary_summary


## 4. Rewrite compact games to a team-perspective table

Each regular-season game becomes exactly two rows—one from each team's perspective. This is a structural transformation, not predictive feature engineering. It prevents winner/loser-column bias and provides a stable foundation for chronological ratings and team-season snapshots.

In [ ]:
def invert_location(series: pd.Series) -> pd.Series:
    return series.map({"H": "A", "A": "H", "N": "N"}).fillna(series)


def compact_to_team_long(frame: pd.DataFrame) -> pd.DataFrame:
    team1 = pd.DataFrame(
        {
            "Gender": frame["Gender"],
            "Season": frame["Season"],
            "DayNum": frame["DayNum"],
            "NumOT": frame["NumOT"],
            "GameKey": frame["GameKey"],
            "TeamID": frame["Team1ID"],
            "OppTeamID": frame["Team2ID"],
            "TeamScore": frame["Team1Score"],
            "OppScore": frame["Team2Score"],
            "Win": frame["Team1Win"].astype("int8"),
            "Margin": frame["Team1Margin"],
            "TeamLoc": frame["Team1Loc"],
        }
    )
    team2 = pd.DataFrame(
        {
            "Gender": frame["Gender"],
            "Season": frame["Season"],
            "DayNum": frame["DayNum"],
            "NumOT": frame["NumOT"],
            "GameKey": frame["GameKey"],
            "TeamID": frame["Team2ID"],
            "OppTeamID": frame["Team1ID"],
            "TeamScore": frame["Team2Score"],
            "OppScore": frame["Team1Score"],
            "Win": (1 - frame["Team1Win"]).astype("int8"),
            "Margin": -frame["Team1Margin"],
            "TeamLoc": invert_location(frame["Team1Loc"]),
        }
    )
    result = pd.concat([team1, team2], ignore_index=True)
    return result.sort_values(
        ["Gender", "Season", "DayNum", "GameKey", "TeamID"]
    ).reset_index(drop=True)


compact_team_long = compact_to_team_long(regular_compact)

rows_per_game = compact_team_long.groupby("GameKey", observed=True).size()
assert rows_per_game.eq(2).all()
assert compact_team_long[["GameKey", "TeamID"]].duplicated().sum() == 0
assert compact_team_long["Win"].isin([0, 1]).all()
assert compact_team_long.groupby("GameKey", observed=True)["Win"].sum().eq(1).all()
assert compact_team_long.groupby("GameKey", observed=True)["Margin"].sum().eq(0).all()

compact_long_path = INTERIM / "team_game_compact_long.parquet"
compact_team_long.to_parquet(compact_long_path, index=False, compression="zstd")

print("Compact team-game rows:", compact_team_long.shape)
print("Unique compact games:", compact_team_long["GameKey"].nunique())
print("Written:", compact_long_path)
compact_team_long.head()

## 5. Carry detailed-data quality flags into modeling

Estimated possessions are calculated here only to diagnose data quality and preserve raw denominators. No offensive/defensive efficiency feature is finalized yet.

The two team possession estimates for a game should be close, but scorekeeping conventions can create discrepancies. Rows above the warning threshold are retained and marked. Later feature experiments will compare full-data, flagged-row-excluded, and robust/winsorized variants.

In [ ]:
POSSESSION_GAP_THRESHOLD = float(
    SPLIT_CONTRACT["quality_thresholds"]["absolute_possession_gap_warning"]
)

required_box_stats = [
    "FGM", "FGA", "FGM3", "FGA3", "FTM", "FTA", "OR", "DR",
    "Ast", "TO", "Stl", "Blk", "PF",
]
for stat in required_box_stats:
    for prefix in ("Team", "Opp"):
        column = f"{prefix}{stat}"
        assert column in regular_detailed.columns, f"Missing detailed field: {column}"

regular_detailed["TeamPossessionsEstimate"] = (
    regular_detailed["TeamFGA"]
    - regular_detailed["TeamOR"]
    + regular_detailed["TeamTO"]
    + 0.475 * regular_detailed["TeamFTA"]
)
regular_detailed["OppPossessionsEstimate"] = (
    regular_detailed["OppFGA"]
    - regular_detailed["OppOR"]
    + regular_detailed["OppTO"]
    + 0.475 * regular_detailed["OppFTA"]
)
regular_detailed["GamePossessionsEstimate"] = (
    regular_detailed["TeamPossessionsEstimate"]
    + regular_detailed["OppPossessionsEstimate"]
) / 2.0
regular_detailed["PossessionGap"] = (
    regular_detailed["TeamPossessionsEstimate"]
    - regular_detailed["OppPossessionsEstimate"]
)
regular_detailed["AbsolutePossessionGap"] = regular_detailed["PossessionGap"].abs()
regular_detailed["PossessionGapFlag"] = regular_detailed["AbsolutePossessionGap"].gt(
    POSSESSION_GAP_THRESHOLD
)
regular_detailed["DetailedDataAvailable"] = True

assert regular_detailed[["GameKey", "TeamID"]].duplicated().sum() == 0
assert regular_detailed.groupby("GameKey", observed=True).size().eq(2).all()
assert regular_detailed.groupby("GameKey", observed=True)["PossessionGap"].sum().abs().lt(1e-9).all()

quality_by_season = (
    regular_detailed.groupby(["Gender", "Season"], observed=True)
    .agg(
        DetailedTeamRows=("TeamID", "size"),
        DetailedGames=("GameKey", "nunique"),
        FlaggedTeamRows=("PossessionGapFlag", "sum"),
        MedianAbsolutePossessionGap=("AbsolutePossessionGap", "median"),
        P95AbsolutePossessionGap=("AbsolutePossessionGap", lambda s: s.quantile(0.95)),
        MaxAbsolutePossessionGap=("AbsolutePossessionGap", "max"),
    )
    .reset_index()
)
quality_by_season["FlaggedTeamRowRate"] = (
    quality_by_season["FlaggedTeamRows"] / quality_by_season["DetailedTeamRows"]
)
quality_by_season.to_csv(
    MODELING_REPORTS / "possession_quality_by_season.csv", index=False
)

quality_by_season.tail(12)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for gender, group in quality_by_season.groupby("Gender", observed=True):
    ax.plot(group["Season"], group["FlaggedTeamRowRate"], marker="o", label=gender)
ax.set_title("Flagged possession-gap rate by season")
ax.set_xlabel("Tournament season")
ax.set_ylabel("Flagged detailed team-row rate")
ax.legend(title="Gender")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_possession_gap_rate_by_season.png", dpi=160, bbox_inches="tight")
plt.show()

## 5A. Game-level possession-gap sensitivity

The team-long table contains two mirrored rows per physical game. The next diagnostic collapses those mirrors to one game and reports several thresholds. The threshold of 7 remains the formal warning threshold, but later feature ablations should compare at least:

- all detailed games;
- exclusion of `gap > 7` games;
- robust/winsorized aggregation without deletion.

This guards against choosing a threshold because it improves a model score.


In [ ]:
game_quality_flags = (
    regular_detailed
    .sort_values(["Gender", "Season", "DayNum", "GameKey", "TeamID"])
    .drop_duplicates("GameKey", keep="first")
    [[
        "Gender", "Season", "DayNum", "GameKey", "TeamID", "OppTeamID",
        "TeamPossessionsEstimate", "OppPossessionsEstimate",
        "GamePossessionsEstimate", "AbsolutePossessionGap", "PossessionGapFlag",
    ]]
    .reset_index(drop=True)
)

for threshold in (3, 5, 7, 10, 15):
    game_quality_flags[f"GapGT{threshold}"] = game_quality_flags[
        "AbsolutePossessionGap"
    ].gt(threshold)

possession_threshold_records: list[dict[str, Any]] = []
for (gender, season), group in game_quality_flags.groupby(
    ["Gender", "Season"], observed=True
):
    row: dict[str, Any] = {
        "Gender": gender,
        "Season": int(season),
        "DetailedGames": len(group),
        "MedianGap": float(group["AbsolutePossessionGap"].median()),
        "P95Gap": float(group["AbsolutePossessionGap"].quantile(0.95)),
        "P99Gap": float(group["AbsolutePossessionGap"].quantile(0.99)),
        "MaxGap": float(group["AbsolutePossessionGap"].max()),
    }
    for threshold in (3, 5, 7, 10, 15):
        count = int(group[f"GapGT{threshold}"].sum())
        row[f"GapGT{threshold}Games"] = count
        row[f"GapGT{threshold}Rate"] = count / len(group) if len(group) else np.nan
    possession_threshold_records.append(row)

possession_thresholds = pd.DataFrame(possession_threshold_records).sort_values(
    ["Gender", "Season"]
).reset_index(drop=True)

game_quality_path = INTERIM / "detailed_regular_game_quality_flags.parquet"
game_quality_flags.to_parquet(
    game_quality_path,
    index=False,
    compression="zstd",
)
possession_thresholds.to_csv(
    MODELING_REPORTS / "possession_gap_threshold_sensitivity.csv",
    index=False,
)

print("Physical detailed regular-season games:", f"{len(game_quality_flags):,}")
print("Written:", game_quality_path)
possession_thresholds.tail(20)


## 6. Build the deterministic pre-tournament base snapshot

The snapshot has exactly one row per `(Gender, Season, TeamID)` and contains only:

- regular-season availability and location counts;
- raw score, margin, overtime, and detailed box-score totals;
- detailed-data coverage and quality indicators;
- team name and same-season conference metadata.

It intentionally stores raw numerators and denominators. Rate definitions, opponent adjustments, empirical priors, scaling, imputation, and feature selection are deferred to notebook `02` and later fold-aware code.

In [ ]:
SNAPSHOT_KEY = ["Gender", "Season", "TeamID"]

compact_snapshot = (
    compact_team_long.groupby(SNAPSHOT_KEY, observed=True)
    .agg(
        CompactGames=("GameKey", "nunique"),
        Wins=("Win", "sum"),
        PointsForSum=("TeamScore", "sum"),
        PointsAgainstSum=("OppScore", "sum"),
        MarginSum=("Margin", "sum"),
        MarginSquaredSum=("Margin", lambda s: float(np.square(s.astype(float)).sum())),
        PositiveMarginGames=("Margin", lambda s: int(s.gt(0).sum())),
        OnePossessionGames=("Margin", lambda s: int(s.abs().le(3).sum())),
        OvertimeGames=("NumOT", lambda s: int(s.gt(0).sum())),
        OvertimePeriodsSum=("NumOT", "sum"),
        HomeGames=("TeamLoc", lambda s: int(s.eq("H").sum())),
        AwayGames=("TeamLoc", lambda s: int(s.eq("A").sum())),
        NeutralGames=("TeamLoc", lambda s: int(s.eq("N").sum())),
        FirstGameDay=("DayNum", "min"),
        LastGameDay=("DayNum", "max"),
    )
    .reset_index()
)
compact_snapshot["Losses"] = compact_snapshot["CompactGames"] - compact_snapshot["Wins"]
compact_snapshot["SnapshotDayNum"] = CUTOFF_DAY

box_sum_columns = [
    f"{side}{stat}"
    for side in ("Team", "Opp")
    for stat in required_box_stats
]

detail_named_aggs: dict[str, tuple[str, Any]] = {
    "DetailedGames": ("GameKey", "nunique"),
    "DetailedFirstGameDay": ("DayNum", "min"),
    "DetailedLastGameDay": ("DayNum", "max"),
    "TeamPossessionsEstimateSum": ("TeamPossessionsEstimate", "sum"),
    "OppPossessionsEstimateSum": ("OppPossessionsEstimate", "sum"),
    "GamePossessionsEstimateSum": ("GamePossessionsEstimate", "sum"),
    "PossessionGapSum": ("PossessionGap", "sum"),
    "AbsolutePossessionGapSum": ("AbsolutePossessionGap", "sum"),
    "MaxAbsolutePossessionGap": ("AbsolutePossessionGap", "max"),
    "FlaggedPossessionGames": ("PossessionGapFlag", "sum"),
}
for column in box_sum_columns:
    detail_named_aggs[f"{column}Sum"] = (column, "sum")

detailed_snapshot = (
    regular_detailed.groupby(SNAPSHOT_KEY, observed=True)
    .agg(**detail_named_aggs)
    .reset_index()
)

snapshot = compact_snapshot.merge(
    detailed_snapshot,
    on=SNAPSHOT_KEY,
    how="left",
    validate="one_to_one",
)

snapshot["DetailedGames"] = snapshot["DetailedGames"].fillna(0).astype("int32")
snapshot["MissingDetailedGames"] = snapshot["CompactGames"] - snapshot["DetailedGames"]
assert snapshot["MissingDetailedGames"].ge(0).all(), (
    "Detailed-game counts exceed compact-game counts for at least one team-season."
)
snapshot["DetailedCoverageRate"] = (
    snapshot["DetailedGames"] / snapshot["CompactGames"].replace(0, np.nan)
)
snapshot["HasAnyDetailedData"] = snapshot["DetailedGames"].gt(0)
snapshot["HasCompleteDetailedCoverage"] = snapshot["DetailedGames"].eq(snapshot["CompactGames"])
snapshot["FlaggedPossessionGames"] = snapshot["FlaggedPossessionGames"].fillna(0).astype("int32")
snapshot["FlaggedPossessionRate"] = (
    snapshot["FlaggedPossessionGames"] / snapshot["DetailedGames"].replace(0, np.nan)
)
snapshot["RichFeatureCoverageEligible"] = snapshot["DetailedCoverageRate"].ge(
    SPLIT_CONTRACT["quality_thresholds"]["minimum_detailed_coverage_for_rich_features"]
)

team_lookup_columns = [c for c in ["Gender", "TeamID", "TeamName", "FirstD1Season", "LastD1Season"] if c in teams.columns]
snapshot = snapshot.merge(
    teams[team_lookup_columns].drop_duplicates(["Gender", "TeamID"]),
    on=["Gender", "TeamID"],
    how="left",
    validate="many_to_one",
)

conference_lookup = team_conferences[["Gender", "Season", "TeamID", "ConfAbbrev"]].drop_duplicates()
assert conference_lookup[["Gender", "Season", "TeamID"]].duplicated().sum() == 0
snapshot = snapshot.merge(
    conference_lookup,
    on=["Gender", "Season", "TeamID"],
    how="left",
    validate="one_to_one",
)

snapshot["SnapshotKey"] = (
    snapshot["Gender"].astype(str)
    + "_"
    + snapshot["Season"].astype(str)
    + "_"
    + snapshot["TeamID"].astype(str)
)

assert snapshot[SNAPSHOT_KEY].duplicated().sum() == 0
assert snapshot["SnapshotKey"].is_unique
assert snapshot["SnapshotDayNum"].eq(CUTOFF_DAY).all()
assert snapshot["LastGameDay"].le(CUTOFF_DAY).all()
assert snapshot["CompactGames"].gt(0).all()
assert snapshot["Wins"].add(snapshot["Losses"]).eq(snapshot["CompactGames"]).all()
assert snapshot[["HomeGames", "AwayGames", "NeutralGames"]].sum(axis=1).eq(snapshot["CompactGames"]).all()

snapshot_path = INTERIM / "team_season_snapshot_base.parquet"
snapshot.to_parquet(snapshot_path, index=False, compression="zstd")

print("Team-season snapshots:", snapshot.shape)
print("Unique seasons:", snapshot["Season"].nunique())
print("Written:", snapshot_path)
snapshot.head()

## 7. Coverage diagnostics for tournament teams

Every historical tournament team should join to a same-season compact snapshot. Rich-feature eligibility is separately measured rather than assumed. This distinction prevents missing detailed rows from being silently converted into legitimate zeros.

In [ ]:
def tournament_team_registry(targets: pd.DataFrame) -> pd.DataFrame:
    team1 = targets[["Gender", "Season", "GameKey", "Team1ID"]].rename(columns={"Team1ID": "TeamID"})
    team2 = targets[["Gender", "Season", "GameKey", "Team2ID"]].rename(columns={"Team2ID": "TeamID"})
    return (
        pd.concat([team1, team2], ignore_index=True)
        .drop_duplicates(["Gender", "Season", "TeamID"])
        .sort_values(["Gender", "Season", "TeamID"])
        .reset_index(drop=True)
    )


tournament_teams = tournament_team_registry(tournament_targets)
coverage_columns = [
    "Gender", "Season", "TeamID", "CompactGames", "DetailedGames",
    "DetailedCoverageRate", "RichFeatureCoverageEligible", "TeamName", "ConfAbbrev",
]
target_team_coverage = tournament_teams.merge(
    snapshot[coverage_columns],
    on=["Gender", "Season", "TeamID"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
target_team_coverage["CompactSnapshotAvailable"] = target_team_coverage["_merge"].eq("both")
target_team_coverage = target_team_coverage.drop(columns="_merge")

assert target_team_coverage["CompactSnapshotAvailable"].all(), (
    "At least one historical tournament team lacks a same-season compact snapshot."
)

coverage_by_season = (
    target_team_coverage.groupby(["Gender", "Season"], observed=True)
    .agg(
        TournamentTeams=("TeamID", "nunique"),
        CompactSnapshotCoverage=("CompactSnapshotAvailable", "mean"),
        MeanDetailedCoverage=("DetailedCoverageRate", "mean"),
        MinDetailedCoverage=("DetailedCoverageRate", "min"),
        RichEligibleTeams=("RichFeatureCoverageEligible", "sum"),
    )
    .reset_index()
)
coverage_by_season["RichEligibleTeamRate"] = (
    coverage_by_season["RichEligibleTeams"] / coverage_by_season["TournamentTeams"]
)

coverage_by_season.to_csv(MODELING_REPORTS / "snapshot_coverage_by_season.csv", index=False)
target_team_coverage.to_csv(MODELING_REPORTS / "target_team_coverage_audit.csv", index=False)

coverage_by_season.tail(16)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for gender, group in coverage_by_season.groupby("Gender", observed=True):
    ax.plot(group["Season"], group["MeanDetailedCoverage"], marker="o", label=gender)
ax.axhline(
    SPLIT_CONTRACT["quality_thresholds"]["minimum_detailed_coverage_for_rich_features"],
    linestyle="--",
    linewidth=1.2,
    label="rich-feature eligibility threshold",
)
ax.set_title("Detailed-data coverage among historical NCAA tournament teams")
ax.set_xlabel("Tournament season")
ax.set_ylabel("Mean detailed coverage")
ax.set_ylim(-0.02, 1.02)
ax.legend(title="Series")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_tournament_team_detailed_coverage.png", dpi=160, bbox_inches="tight")
plt.show()

## 7A. Feature-source availability by gender and season

A state-of-the-art pipeline cannot treat unavailable sources as ordinary numeric zeros. This table records when the optional source families actually exist:

- detailed box scores;
- tournament seeds;
- conference membership;
- game-city coverage;
- men-only Massey systems;
- men-only coach records.

The pooled challenger will be restricted to common features plus explicit availability indicators. The separate men's model may later use time-valid Massey and coach features; the women's model must not receive invented substitutes disguised as real rankings.


In [ ]:
optional_file_map = {
    "massey_ordinals_men": INTERIM / "massey_ordinals_men.parquet",
    "coaches_men": INTERIM / "coaches_men.parquet",
    "game_cities": INTERIM / "game_cities.parquet",
}
optional_frames = {
    name: pd.read_parquet(path) if path.exists() else None
    for name, path in optional_file_map.items()
}

feature_source_availability = (
    snapshot.groupby(["Gender", "Season"], observed=True)
    .agg(
        TeamSeasons=("TeamID", "nunique"),
        CompactTeamGames=("CompactGames", "sum"),
        DetailedTeamGames=("DetailedGames", "sum"),
        TeamsWithConference=("ConfAbbrev", lambda s: int(s.notna().sum())),
        TeamsRichCoverageEligible=("RichFeatureCoverageEligible", "sum"),
    )
    .reset_index()
)
feature_source_availability["DetailedCoverageRate"] = (
    feature_source_availability["DetailedTeamGames"]
    / feature_source_availability["CompactTeamGames"].replace(0, np.nan)
)

seed_availability = (
    seeds.groupby(["Gender", "Season"], observed=True)
    .agg(SeededTeams=("TeamID", "nunique"))
    .reset_index()
)
feature_source_availability = feature_source_availability.merge(
    seed_availability,
    on=["Gender", "Season"],
    how="left",
    validate="one_to_one",
)

massey = optional_frames["massey_ordinals_men"]
if massey is not None:
    massey_valid = massey.loc[massey["RankingDayNum"].le(133)].copy()
    massey_summary = (
        massey_valid.groupby("Season", observed=True)
        .agg(
            MasseyRows=("OrdinalRank", "size"),
            MasseyTeams=("TeamID", "nunique"),
            MasseySystems=("SystemName", "nunique"),
            LatestMasseyDay=("RankingDayNum", "max"),
        )
        .reset_index()
    )
    massey_summary["Gender"] = "M"
    feature_source_availability = feature_source_availability.merge(
        massey_summary,
        on=["Gender", "Season"],
        how="left",
        validate="one_to_one",
    )
else:
    for column in ("MasseyRows", "MasseyTeams", "MasseySystems", "LatestMasseyDay"):
        feature_source_availability[column] = np.nan

coaches = optional_frames["coaches_men"]
if coaches is not None:
    coaches_at_cutoff = coaches.loc[
        coaches["FirstDayNum"].le(CUTOFF_DAY)
        & coaches["LastDayNum"].ge(CUTOFF_DAY)
    ].copy()
    coach_summary = (
        coaches_at_cutoff.groupby("Season", observed=True)
        .agg(
            CoachRowsAtCutoff=("TeamID", "size"),
            TeamsWithCoachAtCutoff=("TeamID", "nunique"),
        )
        .reset_index()
    )
    coach_summary["Gender"] = "M"
    feature_source_availability = feature_source_availability.merge(
        coach_summary,
        on=["Gender", "Season"],
        how="left",
        validate="one_to_one",
    )
else:
    feature_source_availability["CoachRowsAtCutoff"] = np.nan
    feature_source_availability["TeamsWithCoachAtCutoff"] = np.nan

game_city_frame = optional_frames["game_cities"]
if game_city_frame is not None:
    city_regular = game_city_frame.copy()
    if "CRType" in city_regular.columns:
        city_regular = city_regular.loc[city_regular["CRType"].eq("Regular")]
    city_summary = (
        city_regular.groupby(["Gender", "Season"], observed=True)
        .agg(GameCityRows=("CityID", "size"), UniqueGameCities=("CityID", "nunique"))
        .reset_index()
    )
    feature_source_availability = feature_source_availability.merge(
        city_summary,
        on=["Gender", "Season"],
        how="left",
        validate="one_to_one",
    )
else:
    feature_source_availability["GameCityRows"] = np.nan
    feature_source_availability["UniqueGameCities"] = np.nan

feature_source_availability = feature_source_availability.sort_values(
    ["Gender", "Season"]
).reset_index(drop=True)
feature_source_availability.to_csv(
    MODELING_REPORTS / "feature_source_availability.csv",
    index=False,
)
feature_source_availability.tail(24)


## 8. Create the immutable modeling-target registry

Each historical NCAA game receives a role and feature-universe eligibility. The lower-TeamID orientation from notebook `00` is preserved.

- `development`: eligible for rolling-origin model development through 2021;
- `locked_benchmark`: seasons 2022–2025, excluded from tuning and architecture selection;
- `excluded`: structurally unavailable tournament seasons such as 2020.

The 2026 Stage 2 rows are inference requests, not labeled targets.

In [ ]:
target_registry = tournament_targets.copy()
target_registry["TargetKey"] = target_registry["GameKey"]
target_registry["DatasetRole"] = np.select(
    [
        target_registry["Season"].isin(LOCKED_BENCHMARK),
        target_registry["Season"].isin(EXCLUDED_TARGET_SEASONS),
        target_registry["Season"].lt(min(LOCKED_BENCHMARK)),
    ],
    ["locked_benchmark", "excluded", "development"],
    default="future_or_unassigned",
)

def first_season_for(universe: str, gender: str) -> int:
    return int(SPLIT_CONTRACT["universes"][universe][gender]["first_season"])


target_registry["CompactUniverseEligible"] = [
    season >= first_season_for("compact", gender)
    for gender, season in zip(target_registry["Gender"], target_registry["Season"], strict=True)
]
target_registry["RichUniverseEligible"] = [
    season >= first_season_for("rich", gender)
    for gender, season in zip(target_registry["Gender"], target_registry["Season"], strict=True)
]
target_registry["PrimaryModelRoute"] = target_registry["Gender"].map(
    {"M": "men_specific", "W": "women_specific"}
)
target_registry["PooledChallengerEligible"] = target_registry["RichUniverseEligible"]

assert target_registry["TargetKey"].is_unique
assert target_registry["DatasetRole"].ne("future_or_unassigned").all()
assert target_registry.loc[target_registry["DatasetRole"].eq("development"), "Season"].lt(2022).all()
assert target_registry.loc[target_registry["DatasetRole"].eq("locked_benchmark"), "Season"].isin(LOCKED_BENCHMARK).all()

registry_path = INTERIM / "modeling_target_registry.parquet"
target_registry.to_parquet(registry_path, index=False, compression="zstd")

role_summary = (
    target_registry.groupby(["DatasetRole", "Gender"], observed=True)
    .agg(
        Games=("TargetKey", "nunique"),
        Seasons=("Season", "nunique"),
        MinSeason=("Season", "min"),
        MaxSeason=("Season", "max"),
        Team1WinRate=("Team1Win", "mean"),
    )
    .reset_index()
)
role_summary.to_csv(MODELING_REPORTS / "target_registry_summary.csv", index=False)
role_summary

## 9. Generate expanding-window outer folds

A season is the indivisible validation unit. For validation season `Y`, training consists only of eligible tournament seasons earlier than `Y`.

The notebook creates:

- long-history compact folds for seed/Elo baselines;
- rich men's folds;
- rich women's folds;
- aligned pooled rich folds for a fair pooled-versus-separate comparison.

No random row-level split is created.

In [ ]:
def eligible_development_seasons(
    registry: pd.DataFrame,
    *,
    universe: str,
    gender: str,
) -> list[int]:
    eligibility_column = "CompactUniverseEligible" if universe == "compact" else "RichUniverseEligible"
    subset = registry.loc[
        registry["DatasetRole"].eq("development")
        & registry[eligibility_column]
        & registry["Gender"].eq(gender)
    ]
    return sorted(int(value) for value in subset["Season"].unique())


def count_target_rows(
    registry: pd.DataFrame,
    *,
    genders: list[str],
    seasons: list[int],
    universe: str,
) -> int:
    eligibility_column = "CompactUniverseEligible" if universe == "compact" else "RichUniverseEligible"
    mask = (
        registry["Gender"].isin(genders)
        & registry["Season"].isin(seasons)
        & registry[eligibility_column]
    )
    return int(mask.sum())


def build_separate_outer_folds(
    registry: pd.DataFrame,
    *,
    universe: str,
    gender: str,
) -> list[dict[str, Any]]:
    seasons = eligible_development_seasons(registry, universe=universe, gender=gender)
    minimum_prior = int(
        SPLIT_CONTRACT["universes"][universe][gender]["minimum_prior_training_seasons"]
    )
    folds: list[dict[str, Any]] = []
    for validation_season in seasons:
        training_seasons = [season for season in seasons if season < validation_season]
        if len(training_seasons) < minimum_prior:
            continue
        folds.append(
            {
                "OuterFoldID": f"{universe}_{gender}_{validation_season}",
                "Architecture": "separate_gender",
                "Universe": universe,
                "Gender": gender,
                "ValidationSeason": validation_season,
                "TrainingSeasonMin": min(training_seasons),
                "TrainingSeasonMax": max(training_seasons),
                "TrainingSeasonCount": len(training_seasons),
                "TrainingSeasonsJSON": json.dumps(training_seasons),
                "TrainingTargetRows": count_target_rows(
                    registry,
                    genders=[gender],
                    seasons=training_seasons,
                    universe=universe,
                ),
                "ValidationTargetRows": count_target_rows(
                    registry,
                    genders=[gender],
                    seasons=[validation_season],
                    universe=universe,
                ),
            }
        )
    return folds


outer_records: list[dict[str, Any]] = []
for universe in ("compact", "rich"):
    for gender in ("M", "W"):
        outer_records.extend(
            build_separate_outer_folds(
                target_registry,
                universe=universe,
                gender=gender,
            )
        )

# Pooled common-feature folds use validation seasons available to both rich universes.
men_rich_seasons = set(eligible_development_seasons(target_registry, universe="rich", gender="M"))
women_rich_seasons = set(eligible_development_seasons(target_registry, universe="rich", gender="W"))
common_rich_seasons = sorted(men_rich_seasons.intersection(women_rich_seasons))
for validation_season in common_rich_seasons:
    # The validation comparison starts only after the women's rich universe has enough history.
    women_prior = [season for season in women_rich_seasons if season < validation_season]
    if len(women_prior) < int(
        SPLIT_CONTRACT["universes"]["rich"]["W"]["minimum_prior_training_seasons"]
    ):
        continue
    men_training_seasons = sorted(season for season in men_rich_seasons if season < validation_season)
    women_training_seasons = sorted(season for season in women_rich_seasons if season < validation_season)
    pooled_training_seasons = sorted(set(men_training_seasons).union(women_training_seasons))
    outer_records.append(
        {
            "OuterFoldID": f"rich_Pooled_{validation_season}",
            "Architecture": "pooled_common",
            "Universe": "rich",
            "Gender": "Pooled",
            "ValidationSeason": validation_season,
            "TrainingSeasonMin": min(pooled_training_seasons),
            "TrainingSeasonMax": max(pooled_training_seasons),
            "TrainingSeasonCount": len(pooled_training_seasons),
            "TrainingSeasonsJSON": json.dumps(pooled_training_seasons),
            "MenTrainingSeasonsJSON": json.dumps(men_training_seasons),
            "WomenTrainingSeasonsJSON": json.dumps(women_training_seasons),
            "TrainingTargetRows": count_target_rows(
                target_registry,
                genders=["M", "W"],
                seasons=pooled_training_seasons,
                universe="rich",
            ),
            "ValidationTargetRows": count_target_rows(
                target_registry,
                genders=["M", "W"],
                seasons=[validation_season],
                universe="rich",
            ),
        }
    )

outer_folds = pd.DataFrame(outer_records).sort_values(
    ["Universe", "Architecture", "Gender", "ValidationSeason"]
).reset_index(drop=True)

assert not outer_folds.empty
assert outer_folds["OuterFoldID"].is_unique
assert outer_folds["ValidationSeason"].lt(min(LOCKED_BENCHMARK)).all()
assert not outer_folds["ValidationSeason"].isin(EXCLUDED_TARGET_SEASONS).any()
assert outer_folds["TrainingSeasonMax"].lt(outer_folds["ValidationSeason"]).all()
assert outer_folds["TrainingTargetRows"].gt(0).all()
assert outer_folds["ValidationTargetRows"].gt(0).all()

outer_path = INTERIM / "fold_manifest_outer.parquet"
outer_folds.to_parquet(outer_path, index=False, compression="zstd")
outer_folds.to_csv(MODELING_REPORTS / "outer_fold_summary.csv", index=False)

outer_folds.groupby(["Universe", "Architecture", "Gender"], observed=True).agg(
    OuterFolds=("OuterFoldID", "nunique"),
    FirstValidationSeason=("ValidationSeason", "min"),
    LastValidationSeason=("ValidationSeason", "max"),
    MinimumTrainingSeasons=("TrainingSeasonCount", "min"),
    MaximumTrainingRows=("TrainingTargetRows", "max"),
).reset_index()

In [ ]:
plot_data = outer_folds.loc[outer_folds["Universe"].eq("rich")].copy()
fig, ax = plt.subplots(figsize=(12, 6))
for label, group in plot_data.groupby(["Architecture", "Gender"], observed=True):
    display_label = " / ".join(label)
    ax.plot(
        group["ValidationSeason"],
        group["TrainingSeasonCount"],
        marker="o",
        label=display_label,
    )
ax.set_title("Rich-universe expanding-window geometry")
ax.set_xlabel("Outer validation season")
ax.set_ylabel("Number of prior training seasons")
ax.legend(title="Architecture / Gender")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_rich_outer_fold_geometry.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Generate nested inner folds for tuning and model selection

For every outer fold, inner validation seasons are selected only from that outer fold's training history. Later Optuna searches, feature-block selection, imputation, scaling, calibration choice, and ensemble-weight fitting must use these inner folds.

The outer validation season remains untouched until the inner decisions for that fold are complete.

In [ ]:
def parse_seasons(value: str | float | None) -> list[int]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    return [int(v) for v in json.loads(value)]


def validation_rows_for_outer_context(
    registry: pd.DataFrame,
    outer_row: pd.Series,
    validation_season: int,
) -> int:
    genders = ["M", "W"] if outer_row["Gender"] == "Pooled" else [str(outer_row["Gender"])]
    return count_target_rows(
        registry,
        genders=genders,
        seasons=[validation_season],
        universe=str(outer_row["Universe"]),
    )


inner_records: list[dict[str, Any]] = []
max_inner = int(SPLIT_CONTRACT["nested_tuning"]["maximum_inner_validation_seasons"])

for _, outer in outer_folds.iterrows():
    outer_training = parse_seasons(outer["TrainingSeasonsJSON"])
    gender_key = str(outer["Gender"])
    min_prior = int(
        SPLIT_CONTRACT["nested_tuning"]["minimum_inner_prior_seasons"][gender_key]
    )

    candidates = []
    for inner_validation in outer_training:
        inner_training = [season for season in outer_training if season < inner_validation]
        if len(inner_training) >= min_prior:
            candidates.append((inner_validation, inner_training))

    # Use the most recent valid inner folds because they most closely resemble the outer deployment date.
    candidates = candidates[-max_inner:]

    for inner_number, (inner_validation, inner_training) in enumerate(candidates, start=1):
        genders = ["M", "W"] if gender_key == "Pooled" else [gender_key]
        inner_records.append(
            {
                "OuterFoldID": outer["OuterFoldID"],
                "InnerFoldID": f"{outer['OuterFoldID']}_inner_{inner_validation}",
                "InnerFoldNumber": inner_number,
                "Architecture": outer["Architecture"],
                "Universe": outer["Universe"],
                "Gender": gender_key,
                "OuterValidationSeason": int(outer["ValidationSeason"]),
                "InnerValidationSeason": inner_validation,
                "InnerTrainingSeasonMin": min(inner_training),
                "InnerTrainingSeasonMax": max(inner_training),
                "InnerTrainingSeasonCount": len(inner_training),
                "InnerTrainingSeasonsJSON": json.dumps(inner_training),
                "InnerTrainingTargetRows": count_target_rows(
                    target_registry,
                    genders=genders,
                    seasons=inner_training,
                    universe=str(outer["Universe"]),
                ),
                "InnerValidationTargetRows": validation_rows_for_outer_context(
                    target_registry,
                    outer,
                    inner_validation,
                ),
            }
        )

inner_folds = pd.DataFrame(inner_records).sort_values(
    ["Universe", "Architecture", "Gender", "OuterValidationSeason", "InnerValidationSeason"]
).reset_index(drop=True)

assert not inner_folds.empty
assert inner_folds["InnerFoldID"].is_unique
assert inner_folds["InnerValidationSeason"].lt(inner_folds["OuterValidationSeason"]).all()
assert inner_folds["InnerTrainingSeasonMax"].lt(inner_folds["InnerValidationSeason"]).all()
assert inner_folds["InnerTrainingTargetRows"].gt(0).all()
assert inner_folds["InnerValidationTargetRows"].gt(0).all()
assert inner_folds["OuterValidationSeason"].lt(min(LOCKED_BENCHMARK)).all()

inner_path = INTERIM / "fold_manifest_inner.parquet"
inner_folds.to_parquet(inner_path, index=False, compression="zstd")
inner_folds.to_csv(MODELING_REPORTS / "inner_fold_summary.csv", index=False)

inner_folds.groupby(["Universe", "Architecture", "Gender"], observed=True).agg(
    OuterFoldsWithInnerCV=("OuterFoldID", "nunique"),
    TotalInnerFolds=("InnerFoldID", "nunique"),
    MinimumInnerTrainingSeasons=("InnerTrainingSeasonCount", "min"),
    MaximumInnerTrainingSeasons=("InnerTrainingSeasonCount", "max"),
).reset_index()

## 10A. Secondary leave-one-season-out benchmark

Public solutions often report leave-one-season-out (LOSO) performance. We preserve it for comparability, but every row explicitly records whether training includes seasons later than the validation season.

LOSO results must never replace the primary rolling-origin estimate in the final report.


In [ ]:
def build_loso_manifest(
    registry: pd.DataFrame,
    *,
    universe: str,
    gender: str,
    architecture: str,
) -> list[dict[str, Any]]:
    if gender == "Pooled":
        genders = ["M", "W"]
        eligible = registry.loc[
            registry["DatasetRole"].eq("development")
            & registry["RichUniverseEligible"]
            & registry["Gender"].isin(genders)
        ]
    else:
        genders = [gender]
        eligibility_column = (
            "CompactUniverseEligible" if universe == "compact" else "RichUniverseEligible"
        )
        eligible = registry.loc[
            registry["DatasetRole"].eq("development")
            & registry[eligibility_column]
            & registry["Gender"].eq(gender)
        ]

    seasons = sorted(int(value) for value in eligible["Season"].unique())
    records: list[dict[str, Any]] = []
    for validation_season in seasons:
        training_seasons = [season for season in seasons if season != validation_season]
        records.append(
            {
                "LOSOFoldID": f"{universe}_{gender}_loso_{validation_season}",
                "Architecture": architecture,
                "Universe": universe,
                "Gender": gender,
                "ValidationSeason": validation_season,
                "TrainingSeasonMin": min(training_seasons),
                "TrainingSeasonMax": max(training_seasons),
                "TrainingSeasonCount": len(training_seasons),
                "TrainingSeasonsJSON": json.dumps(training_seasons),
                "TrainingTargetRows": int(
                    eligible["Season"].isin(training_seasons).sum()
                ),
                "ValidationTargetRows": int(
                    eligible["Season"].eq(validation_season).sum()
                ),
                "UsesFutureRelativeToValidation": any(
                    season > validation_season for season in training_seasons
                ),
                "IntendedUse": "secondary_public_solution_comparison_only",
            }
        )
    return records


loso_records: list[dict[str, Any]] = []
for universe in ("compact", "rich"):
    for gender in ("M", "W"):
        loso_records.extend(
            build_loso_manifest(
                target_registry,
                universe=universe,
                gender=gender,
                architecture="separate_gender",
            )
        )
loso_records.extend(
    build_loso_manifest(
        target_registry,
        universe="rich",
        gender="Pooled",
        architecture="pooled_common",
    )
)

loso_folds = pd.DataFrame(loso_records).sort_values(
    ["Universe", "Architecture", "Gender", "ValidationSeason"]
).reset_index(drop=True)
assert not loso_folds.empty
assert loso_folds["LOSOFoldID"].is_unique
assert loso_folds["ValidationTargetRows"].gt(0).all()
assert not set(loso_folds["ValidationSeason"]).intersection(LOCKED_BENCHMARK)

loso_path = INTERIM / "fold_manifest_secondary_loso.parquet"
loso_folds.to_parquet(loso_path, index=False, compression="zstd")
loso_folds.to_csv(MODELING_REPORTS / "secondary_loso_fold_summary.csv", index=False)

loso_folds.groupby(
    ["Universe", "Architecture", "Gender", "UsesFutureRelativeToValidation"],
    observed=True,
).size().to_frame("Folds").reset_index()


## 11. Locked-benchmark and season-level diagnostics

The benchmark seasons are isolated now, before any model score exists. The report also exposes target volume, class balance, and season drift by gender so later gains cannot be attributed to an accidental change in evaluation population.

In [ ]:
seed_diag = seeds[["Gender", "Season", "TeamID", "SeedNum"]].drop_duplicates()
assert seed_diag[["Gender", "Season", "TeamID"]].duplicated().sum() == 0

seed1 = seed_diag.rename(columns={"TeamID": "Team1ID", "SeedNum": "Team1SeedNum"})
seed2 = seed_diag.rename(columns={"TeamID": "Team2ID", "SeedNum": "Team2SeedNum"})
diagnostic_targets = (
    target_registry.merge(
        seed1,
        on=["Gender", "Season", "Team1ID"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        seed2,
        on=["Gender", "Season", "Team2ID"],
        how="left",
        validate="many_to_one",
    )
)
diagnostic_targets["BothSeedsAvailable"] = (
    diagnostic_targets["Team1SeedNum"].notna()
    & diagnostic_targets["Team2SeedNum"].notna()
)
diagnostic_targets["SeedGap"] = (
    diagnostic_targets["Team1SeedNum"] - diagnostic_targets["Team2SeedNum"]
)
valid_seed_favorite = diagnostic_targets["BothSeedsAvailable"] & diagnostic_targets["SeedGap"].ne(0)
diagnostic_targets["SeedUnderdogWon"] = np.where(
    valid_seed_favorite,
    (
        (diagnostic_targets["SeedGap"].gt(0) & diagnostic_targets["Team1Win"].eq(1))
        | (diagnostic_targets["SeedGap"].lt(0) & diagnostic_targets["Team1Win"].eq(0))
    ).astype(float),
    np.nan,
)

unique_team_counts = (
    tournament_teams.groupby(["Gender", "Season"], observed=True)["TeamID"]
    .nunique()
    .rename("UniqueTeams")
    .reset_index()
)

season_target_diagnostics = (
    diagnostic_targets.groupby(["DatasetRole", "Gender", "Season"], observed=True)
    .agg(
        TournamentGames=("TargetKey", "nunique"),
        Team1WinRate=("Team1Win", "mean"),
        MeanAbsoluteMargin=("Team1Margin", lambda s: s.abs().mean()),
        SeedCoverageRate=("BothSeedsAvailable", "mean"),
        MeanAbsoluteSeedGap=("SeedGap", lambda s: s.abs().mean()),
        SeedUnderdogWinRate=("SeedUnderdogWon", "mean"),
    )
    .reset_index()
    .merge(unique_team_counts, on=["Gender", "Season"], how="left", validate="one_to_one")
)
season_target_diagnostics.to_csv(
    MODELING_REPORTS / "season_target_diagnostics.csv", index=False
)

locked_rows = diagnostic_targets.loc[
    diagnostic_targets["DatasetRole"].eq("locked_benchmark")
]
assert set(locked_rows["Season"].unique()) == LOCKED_BENCHMARK
assert not set(outer_folds["ValidationSeason"]).intersection(LOCKED_BENCHMARK)
assert not set(inner_folds["InnerValidationSeason"]).intersection(LOCKED_BENCHMARK)

for values in outer_folds["TrainingSeasonsJSON"]:
    assert not set(parse_seasons(values)).intersection(LOCKED_BENCHMARK)
for values in inner_folds["InnerTrainingSeasonsJSON"]:
    assert not set(parse_seasons(values)).intersection(LOCKED_BENCHMARK)

locked_summary = (
    locked_rows.groupby(["Gender", "Season"], observed=True)
    .agg(
        Games=("TargetKey", "nunique"),
        Team1WinRate=("Team1Win", "mean"),
        MeanAbsoluteMargin=("Team1Margin", lambda s: s.abs().mean()),
        SeedUnderdogWinRate=("SeedUnderdogWon", "mean"),
    )
    .reset_index()
)
locked_summary

## 11A. Explicit locked-benchmark protocols

Because this competition is already complete and public solution knowledge can influence our design, 2022–2025 are best described as a **locked local benchmark**, not a perfectly pristine external test.

Nevertheless, they are isolated from all future selection in this repository. Two scientifically useful evaluations are pre-registered:

1. **Prequential:** recipe frozen after 2021; for test season `Y`, refit the chosen pipeline on all labeled seasons before `Y` and predict `Y`.
2. **Static block:** fit the tournament-outcome model through 2021 and carry it unchanged across 2022–2025, while still using each season's legal regular-season team snapshot.

No feature family, hyperparameter, calibrator, clipping rule, or ensemble weight may be changed after looking at these scores and still be described as confirmatory.


In [ ]:
def locked_benchmark_records(
    registry: pd.DataFrame,
    *,
    universe: str,
    gender: str,
    architecture: str,
) -> list[dict[str, Any]]:
    if gender == "Pooled":
        genders = ["M", "W"]
        eligibility_column = "RichUniverseEligible"
    else:
        genders = [gender]
        eligibility_column = (
            "CompactUniverseEligible" if universe == "compact" else "RichUniverseEligible"
        )

    eligible = registry.loc[
        registry["Gender"].isin(genders)
        & registry[eligibility_column]
    ].copy()
    static_training_seasons = sorted(
        int(value)
        for value in eligible.loc[
            eligible["Season"].le(DEVELOPMENT_LAST_SEASON), "Season"
        ].unique()
    )

    records: list[dict[str, Any]] = []
    for test_season in sorted(LOCKED_BENCHMARK):
        test_rows = eligible.loc[eligible["Season"].eq(test_season)]
        assert not test_rows.empty, (
            f"No {universe}/{gender} benchmark targets found for {test_season}."
        )

        prequential_training_seasons = sorted(
            int(value)
            for value in eligible.loc[eligible["Season"].lt(test_season), "Season"].unique()
        )
        for mode, training_seasons in (
            ("prequential", prequential_training_seasons),
            ("static_block", static_training_seasons),
        ):
            records.append(
                {
                    "BenchmarkFoldID": f"{universe}_{gender}_{mode}_{test_season}",
                    "Architecture": architecture,
                    "Universe": universe,
                    "Gender": gender,
                    "BenchmarkMode": mode,
                    "TestSeason": test_season,
                    "RecipeFrozenAfterSeason": DEVELOPMENT_LAST_SEASON,
                    "TrainingSeasonMin": min(training_seasons),
                    "TrainingSeasonMax": max(training_seasons),
                    "TrainingSeasonCount": len(training_seasons),
                    "TrainingSeasonsJSON": json.dumps(training_seasons),
                    "TrainingTargetRows": int(
                        eligible["Season"].isin(training_seasons).sum()
                    ),
                    "TestTargetRows": len(test_rows),
                }
            )
    return records


locked_records: list[dict[str, Any]] = []
for universe in ("compact", "rich"):
    for gender in ("M", "W"):
        locked_records.extend(
            locked_benchmark_records(
                target_registry,
                universe=universe,
                gender=gender,
                architecture="separate_gender",
            )
        )
locked_records.extend(
    locked_benchmark_records(
        target_registry,
        universe="rich",
        gender="Pooled",
        architecture="pooled_common",
    )
)

locked_benchmark_folds = pd.DataFrame(locked_records).sort_values(
    ["Universe", "Architecture", "Gender", "BenchmarkMode", "TestSeason"]
).reset_index(drop=True)
assert locked_benchmark_folds["BenchmarkFoldID"].is_unique
assert set(locked_benchmark_folds["TestSeason"]) == LOCKED_BENCHMARK
assert locked_benchmark_folds.loc[
    locked_benchmark_folds["BenchmarkMode"].eq("static_block"),
    "TrainingSeasonMax",
].eq(DEVELOPMENT_LAST_SEASON).all()
assert (
    locked_benchmark_folds.loc[
        locked_benchmark_folds["BenchmarkMode"].eq("prequential"),
        "TrainingSeasonMax",
    ]
    < locked_benchmark_folds.loc[
        locked_benchmark_folds["BenchmarkMode"].eq("prequential"),
        "TestSeason",
    ]
).all()

locked_benchmark_path = INTERIM / "fold_manifest_locked_benchmark.parquet"
locked_benchmark_folds.to_parquet(
    locked_benchmark_path,
    index=False,
    compression="zstd",
)
locked_benchmark_folds.to_csv(
    MODELING_REPORTS / "locked_benchmark_fold_summary.csv",
    index=False,
)
locked_benchmark_folds


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for gender, group in season_target_diagnostics.groupby("Gender", observed=True):
    ax.plot(group["Season"], group["Team1WinRate"], marker="o", label=gender)
ax.axhline(0.5, linestyle="--", linewidth=1.0)
for season in sorted(LOCKED_BENCHMARK):
    ax.axvspan(season - 0.45, season + 0.45, alpha=0.08)
ax.set_title("Lower-TeamID win rate by tournament season; shaded seasons are locked benchmark")
ax.set_xlabel("Tournament season")
ax.set_ylabel("P(lower-TeamID team wins)")
ax.legend(title="Gender")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_target_balance_by_season.png", dpi=160, bbox_inches="tight")
plt.show()

## 11B. Model architecture comparison contract

The final system will not choose separate or pooled models by narrative preference. The common-feature comparison is matched on the same recent development seasons. A partial-pooling blend is eligible only if it improves nested out-of-fold Brier score and remains stable across seasons.


In [ ]:
matched_architecture_seasons = sorted(
    set(
        outer_folds.loc[
            (outer_folds["Universe"].eq("rich"))
            & (outer_folds["Gender"].eq("Pooled")),
            "ValidationSeason",
        ]
    )
)

architecture_contract = pd.DataFrame(
    [
        {
            "Architecture": "separate_men",
            "Role": "primary",
            "TrainingPopulation": "men only",
            "FeatureUniverse": "common + men_only",
            "Calibration": "men-specific out-of-fold only",
            "BlendWeights": "men-specific nested OOF only",
            "MatchedComparisonSeasonsJSON": json.dumps(matched_architecture_seasons),
        },
        {
            "Architecture": "separate_women",
            "Role": "primary",
            "TrainingPopulation": "women only",
            "FeatureUniverse": "common + women_only",
            "Calibration": "women-specific out-of-fold only",
            "BlendWeights": "women-specific nested OOF only",
            "MatchedComparisonSeasonsJSON": json.dumps(matched_architecture_seasons),
        },
        {
            "Architecture": "pooled_common",
            "Role": "mandatory challenger",
            "TrainingPopulation": "men + women",
            "FeatureUniverse": "common only + explicit gender terms",
            "Calibration": "pooled and gender-conditional variants compared in nested OOF",
            "BlendWeights": "nested OOF only",
            "MatchedComparisonSeasonsJSON": json.dumps(matched_architecture_seasons),
        },
        {
            "Architecture": "partial_pooling",
            "Role": "eligible final challenger",
            "TrainingPopulation": "blend pooled and gender-specific predictions",
            "FeatureUniverse": "prediction-level blend",
            "Calibration": "after constituent OOF predictions only",
            "BlendWeights": "nonnegative, sum-to-one, gender-specific, nested OOF",
            "MatchedComparisonSeasonsJSON": json.dumps(matched_architecture_seasons),
        },
    ]
)
architecture_contract.to_csv(
    MODELING_REPORTS / "architecture_comparison_contract.csv",
    index=False,
)
architecture_contract


## 12. Build Stage 2 routing without generating predictions

The competition's combined submission is not evidence that one combined model is best. Each row is assigned to a gender-specific primary route, while retaining the pooled challenger route for later blending.

The routing table also records whether both teams have a 2026 compact snapshot and whether both have a 2026 seed. A future final notebook can use a seed-aware model when both teams are seeded and a seed-free fallback for all other required hypothetical matchups.

In [ ]:
stage2 = submission_matchups.loc[
    submission_matchups["SubmissionFile"].eq("SampleSubmissionStage2")
].copy()

assert stage2["Season"].eq(TARGET_SEASON).all()
assert stage2["ID"].is_unique
assert (stage2["Team1ID"] < stage2["Team2ID"]).all()
assert stage2["Gender"].isin(["M", "W"]).all()

snapshot_2026_keys = snapshot.loc[
    snapshot["Season"].eq(TARGET_SEASON), ["Gender", "TeamID"]
].drop_duplicates()
snapshot_key_set = set(map(tuple, snapshot_2026_keys.to_numpy()))

seed_2026_keys = seeds.loc[
    seeds["Season"].eq(TARGET_SEASON), ["Gender", "TeamID"]
].drop_duplicates()
seed_key_set = set(map(tuple, seed_2026_keys.to_numpy()))

stage2["PrimaryModelRoute"] = stage2["Gender"].map(
    {"M": "men_specific_final", "W": "women_specific_final"}
)
stage2["PooledChallengerRoute"] = "pooled_common_final"
stage2["Team1SnapshotAvailable"] = [
    (gender, team_id) in snapshot_key_set
    for gender, team_id in zip(stage2["Gender"], stage2["Team1ID"], strict=True)
]
stage2["Team2SnapshotAvailable"] = [
    (gender, team_id) in snapshot_key_set
    for gender, team_id in zip(stage2["Gender"], stage2["Team2ID"], strict=True)
]
stage2["BothSnapshotsAvailable"] = (
    stage2["Team1SnapshotAvailable"] & stage2["Team2SnapshotAvailable"]
)
stage2["Team1SeedAvailable"] = [
    (gender, team_id) in seed_key_set
    for gender, team_id in zip(stage2["Gender"], stage2["Team1ID"], strict=True)
]
stage2["Team2SeedAvailable"] = [
    (gender, team_id) in seed_key_set
    for gender, team_id in zip(stage2["Gender"], stage2["Team2ID"], strict=True)
]
stage2["BothSeedsAvailable"] = stage2["Team1SeedAvailable"] & stage2["Team2SeedAvailable"]
stage2["FeatureRoute"] = np.where(
    stage2["BothSeedsAvailable"],
    "seed_aware",
    "seed_free_fallback",
)

routing_path = INTERIM / "submission_routing.parquet"
stage2.to_parquet(routing_path, index=False, compression="zstd")

routing_summary = (
    stage2.groupby(["Gender", "PrimaryModelRoute", "FeatureRoute"], observed=True)
    .agg(
        Matchups=("ID", "nunique"),
        BothSnapshotsAvailableRate=("BothSnapshotsAvailable", "mean"),
    )
    .reset_index()
)
routing_summary.to_csv(MODELING_REPORTS / "stage2_routing_summary.csv", index=False)
routing_summary

## 13. Global leakage and reproducibility test suite

These assertions are intentionally redundant. A state-of-the-art model built on a leaky experimental contract is still invalid.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


leakage_records: list[tuple[str, bool, str]] = []


def check(name: str, passed: bool, details: str) -> None:
    leakage_records.append((name, bool(passed), details))


# Structural and temporal source checks.
check(
    "regular compact only",
    regular_compact["GameType"].eq("Regular").all(),
    f"rows={len(regular_compact):,}",
)
check(
    "regular detailed only",
    regular_detailed["GameType"].eq("Regular").all(),
    f"rows={len(regular_detailed):,}",
)
check(
    "maximum feature day <= cutoff",
    regular_compact["DayNum"].le(CUTOFF_DAY).all()
    and regular_detailed["DayNum"].le(CUTOFF_DAY).all(),
    f"cutoff={CUTOFF_DAY}",
)
check(
    "target/future NCAA labels quarantined",
    tournament_targets["Season"].lt(TARGET_SEASON).all(),
    f"quarantined_rows={len(quarantined_target_outcomes):,}",
)

# Snapshot checks.
check(
    "unique team-season snapshots",
    snapshot[SNAPSHOT_KEY].duplicated().sum() == 0,
    f"duplicates={snapshot[SNAPSHOT_KEY].duplicated().sum():,}",
)
check(
    "snapshot day fixed at cutoff",
    snapshot["SnapshotDayNum"].eq(CUTOFF_DAY).all(),
    f"cutoff={CUTOFF_DAY}",
)
check(
    "snapshot source games do not exceed cutoff",
    snapshot["LastGameDay"].le(CUTOFF_DAY).all(),
    f"max_last_game_day={snapshot['LastGameDay'].max()}",
)
check(
    "every snapshot has compact exposure",
    snapshot["CompactGames"].gt(0).all(),
    f"zero_game_rows={snapshot['CompactGames'].le(0).sum():,}",
)

# Target orientation and label checks.
check(
    "binary target labels",
    target_registry["Team1Win"].isin([0, 1]).all(),
    f"invalid_rows={(~target_registry['Team1Win'].isin([0, 1])).sum():,}",
)
check(
    "lower-TeamID orientation",
    target_registry["Team1ID"].lt(target_registry["Team2ID"]).all(),
    f"invalid_rows={(~target_registry['Team1ID'].lt(target_registry['Team2ID'])).sum():,}",
)
check(
    "unique target keys",
    target_registry["TargetKey"].is_unique,
    f"duplicates={target_registry['TargetKey'].duplicated().sum():,}",
)
check(
    "historical target teams have compact snapshots",
    target_team_coverage["CompactSnapshotAvailable"].all(),
    f"missing={(~target_team_coverage['CompactSnapshotAvailable']).sum():,}",
)

# Outer fold checks.
outer_time_valid = True
outer_locked_clean = True
for _, row in outer_folds.iterrows():
    training = parse_seasons(row["TrainingSeasonsJSON"])
    outer_time_valid &= bool(
        training == sorted(training)
        and max(training) < int(row["ValidationSeason"])
        and int(row["ValidationSeason"]) not in training
    )
    outer_locked_clean &= not bool(set(training).intersection(LOCKED_BENCHMARK))
check(
    "all outer training seasons precede validation",
    outer_time_valid,
    f"outer_folds={len(outer_folds):,}",
)
check(
    "locked seasons absent from development outer folds",
    outer_locked_clean
    and not set(outer_folds["ValidationSeason"]).intersection(LOCKED_BENCHMARK),
    f"locked={sorted(LOCKED_BENCHMARK)}",
)

# Inner fold checks.
inner_time_valid = True
inner_locked_clean = True
for _, row in inner_folds.iterrows():
    training = parse_seasons(row["InnerTrainingSeasonsJSON"])
    inner_time_valid &= bool(
        training == sorted(training)
        and max(training) < int(row["InnerValidationSeason"])
        and int(row["InnerValidationSeason"]) < int(row["OuterValidationSeason"])
    )
    inner_locked_clean &= not bool(set(training).intersection(LOCKED_BENCHMARK))
check(
    "all inner training seasons precede inner and outer validation",
    inner_time_valid,
    f"inner_folds={len(inner_folds):,}",
)
check(
    "locked seasons absent from development inner folds",
    inner_locked_clean
    and not set(inner_folds["InnerValidationSeason"]).intersection(LOCKED_BENCHMARK),
    f"locked={sorted(LOCKED_BENCHMARK)}",
)

# Secondary and benchmark protocol checks.
check(
    "LOSO manifest explicitly identifies future-relative training",
    loso_folds["UsesFutureRelativeToValidation"].any(),
    "LOSO is secondary only",
)
check(
    "locked benchmark manifest covers every locked season",
    set(locked_benchmark_folds["TestSeason"]) == LOCKED_BENCHMARK,
    f"test_seasons={sorted(locked_benchmark_folds['TestSeason'].unique())}",
)
check(
    "static benchmark training ends in 2021",
    locked_benchmark_folds.loc[
        locked_benchmark_folds["BenchmarkMode"].eq("static_block"),
        "TrainingSeasonMax",
    ].eq(DEVELOPMENT_LAST_SEASON).all(),
    f"development_last_season={DEVELOPMENT_LAST_SEASON}",
)
check(
    "prequential benchmark training precedes test season",
    (
        locked_benchmark_folds.loc[
            locked_benchmark_folds["BenchmarkMode"].eq("prequential"),
            "TrainingSeasonMax",
        ]
        < locked_benchmark_folds.loc[
            locked_benchmark_folds["BenchmarkMode"].eq("prequential"),
            "TestSeason",
        ]
    ).all(),
    "recipe remains frozen after 2021",
)

# Stage 2 routing checks.
check(
    "Stage 2 season and gender routing valid",
    stage2["Season"].eq(TARGET_SEASON).all()
    and stage2["Gender"].isin(["M", "W"]).all()
    and stage2["PrimaryModelRoute"].notna().all(),
    f"rows={len(stage2):,}",
)
check(
    "Stage 2 IDs unique and lower-ID oriented",
    stage2["ID"].is_unique
    and stage2["Team1ID"].lt(stage2["Team2ID"]).all(),
    f"duplicate_ids={stage2['ID'].duplicated().sum():,}",
)

leakage_checks = pd.DataFrame(
    leakage_records,
    columns=["Check", "Passed", "Details"],
)
leakage_checks.to_csv(MODELING_REPORTS / "leakage_checks.csv", index=False)

failed_leakage_checks = leakage_checks.loc[~leakage_checks["Passed"]]
if not failed_leakage_checks.empty:
    display(failed_leakage_checks.reset_index(drop=True))
    raise AssertionError("One or more blocking leakage/reproducibility checks failed.")

output_paths = {
    "compact_team_long": compact_long_path,
    "detailed_regular_game_quality_flags": game_quality_path,
    "team_season_snapshot_base": snapshot_path,
    "modeling_target_registry": registry_path,
    "fold_manifest_outer": outer_path,
    "fold_manifest_inner": inner_path,
    "fold_manifest_secondary_loso": loso_path,
    "fold_manifest_locked_benchmark": locked_benchmark_path,
    "submission_routing": routing_path,
}
output_hashes = {name: sha256_file(path) for name, path in output_paths.items()}

print(leakage_checks.groupby("Passed", observed=True).size())
print("All blocking leakage and reproducibility checks passed.")
leakage_checks


## 14. Final readiness report

The report fingerprints the contract and all generated artifacts. Commit the small reports and `configs/splits.yaml`; keep Parquet artifacts ignored by Git.

In [ ]:
readiness = {
    "notebook": "01_split_protocol_and_pre_tournament_snapshots.ipynb",
    "status": "complete",
    "split_contract_sha256": contract_hash,
    "feature_cutoff_day": CUTOFF_DAY,
    "target_season": TARGET_SEASON,
    "development_last_season": DEVELOPMENT_LAST_SEASON,
    "locked_benchmark_seasons": sorted(LOCKED_BENCHMARK),
    "compact_team_game_rows": int(len(compact_team_long)),
    "detailed_physical_game_quality_rows": int(len(game_quality_flags)),
    "team_season_snapshots": int(len(snapshot)),
    "historical_tournament_targets": int(len(target_registry)),
    "outer_folds": int(len(outer_folds)),
    "inner_folds": int(len(inner_folds)),
    "secondary_loso_folds": int(len(loso_folds)),
    "locked_benchmark_manifest_rows": int(len(locked_benchmark_folds)),
    "stage2_matchups": int(len(stage2)),
    "men_stage2_matchups": int(stage2["Gender"].eq("M").sum()),
    "women_stage2_matchups": int(stage2["Gender"].eq("W").sum()),
    "target_team_compact_snapshot_coverage": float(
        target_team_coverage["CompactSnapshotAvailable"].mean()
    ),
    "stage2_both_snapshot_coverage": float(stage2["BothSnapshotsAvailable"].mean()),
    "quarantined_target_or_future_outcome_rows": int(len(quarantined_target_outcomes)),
    "blocking_leakage_checks_failed": int((~leakage_checks["Passed"]).sum()),
    "artifact_sha256": output_hashes,
}

(MODELING_REPORTS / "01_readiness_summary.json").write_text(
    json.dumps(readiness, indent=2),
    encoding="utf-8",
)

protocol_markdown = f'''# Frozen Modeling and Validation Protocol

## Forecast boundary

- Target season: {TARGET_SEASON}
- Feature cutoff: DayNum {CUTOFF_DAY}
- Same-season NCAA tournament outcomes are forbidden as predictors.
- Any target/future outcomes found in source files are quarantined.

## Architectures

- Primary: separate men's and women's pipelines
- Mandatory challenger: pooled common-feature model
- Eligible final challenger: partial-pooling prediction blend

## Validation

- Primary: nested expanding-window validation by complete tournament season
- Secondary: LOSO only for public-solution comparability
- Development ends: {DEVELOPMENT_LAST_SEASON}

## Locked local benchmark

- Seasons: {sorted(LOCKED_BENCHMARK)}
- Primary mode: prequential refit with the recipe frozen after 2021
- Sensitivity mode: static target model fit through 2021
- Because the competition is complete and public solution knowledge is available, this is not described as a perfectly blind external test.

## Final 2026 replay

After the full recipe is frozen and the locked-benchmark report is issued, refit the selected system using all eligible labeled tournaments through 2025 and score Stage 2 rows through gender-specific routing.
'''
(MODELING_REPORTS / "SPLIT_PROTOCOL.md").write_text(
    protocol_markdown,
    encoding="utf-8",
)

print(json.dumps(readiness, indent=2))
assert readiness["blocking_leakage_checks_failed"] == 0
assert readiness["target_team_compact_snapshot_coverage"] == 1.0

print("\nNOTEBOOK 01 COMPLETE — split contract frozen and pre-tournament snapshots ready.")
print("No predictive feature engineering or model fitting has occurred yet.")


# Stop here and return the executed notebook for review

Do not train a model yet. The outputs from this notebook determine the exact safe universe for feature engineering.

The next notebook will be:

```text
02_state_of_the_art_feature_store.ipynb
```

## Feature engineering begins in notebook `02`

Notebook `02` will build feature blocks under the frozen fold contract:

1. **selection priors:** seed, play-in status, smoothed historical seed-pair priors;
2. **possession and Four Factors:** eFG%, turnover rate, offensive rebounding, free-throw rate, pace;
3. **robust team form:** full-season, exponentially weighted, last-N, volatility, and trend variants;
4. **opponent adjustment:** ridge offense/defense, SRS, Colley, Bradley–Terry, schedule strength, quality wins, bad losses;
5. **dynamic ratings:** standard Elo, margin-aware Elo, uncertainty-aware/Glicko-style ratings, and season carryover selected only inside inner folds;
6. **men-only public rankings:** as-of-day Massey consensus, disagreement, coverage, momentum, and selected stable systems;
7. **context:** home/away/neutral splits, conference strength, rest, schedule density, and coach continuity where available;
8. **matchup representation:** signed differences, selected absolute gaps, basketball-motivated offense-versus-defense interactions, and mirrored-pair invariance tests;
9. **quality and uncertainty:** detailed-data coverage, rating uncertainty, missingness indicators, and full/excluded/robust possession-quality variants;
10. **fold-safe artifacts:** common, men's-only, and women's-only feature stores with no globally learned transformations.

Notebook `03` will establish the model ladder and nested rolling-origin baselines:

```text
0.50 constant
→ seed-only logistic
→ legal current-season Elo logistic
→ seed + Elo logistic
→ rich elastic-net logistic
→ direct-probability XGBoost/LightGBM
→ point-margin regressors + OOF calibration
→ statistical rating challengers
→ small regularized neural challenger
→ constrained, gender-specific calibrated ensemble
```

Upload this notebook after running all cells. Its coverage tables, fold counts, quarantine status, Stage 2 routing, and assertions determine whether feature engineering can begin without revising the data contract.
